
*   $I = \{1, 2, 3\}$ — множество типов исходных брусков:
    *   $1$ — бруски длиной 2.7 м;
    *   $2$ — бруски длиной 3.5 м;
    *   $3$ — бруски длиной 4.1 м.
*   $J$ — множество всех возможных **карт раскроя** (паттернов).
    *   Каждый паттерн $j \in J$ характеризуется кортежем $(l_j, a_j, b_j)$, где:
        *   $l_j \in \{2.7, 3.5, 4.1\}$ — длина исходного бруска;
        *   $a_j$ — количество заготовок длиной 1.0 м, получаемых при раскрое;
        *   $b_j$ — количество заготовок длиной 1.5 м, получаемых при раскрое.
    *   Условие допустимости паттерна: $1.0 \cdot a_j + 1.5 \cdot b_j \le l_j$.


*   $N_1 = 100$ — количество брусков длиной 2.7 м на складе.
*   $N_2 = 200$ — количество брусков длиной 3.5 м на складе.
*   $N_3 = 300$ — количество брусков длиной 4.1 м на складе.
*   $req_1 = 3$ — количество заготовок длиной 1.0 м, необходимых для одной рамы.
*   $req_2 = 2$ — количество заготовок длиной 1.5 м, необходимых для одной рамы.

## 3. Переменные
*   $x_j \ge 0$ (целочисленная) — количество раз, которое применяется паттерн раскроя $j \in J$.
*   $F \ge 0$ (целочисленная) — итоговое количество изготовленных оконных рам.

## 4. Целевая функция
Необходимо максимизировать количество готовых рам:
$$ F \rightarrow max$$

## 5. Ограничения

### Ограничение по запасам исходных брусков
Количество использованных брусков каждого типа не должно превышать их наличие на складе:
$$ \sum_{j \in J : l_j = 2.7} x_j \le 100 $$
$$ \sum_{j \in J : l_j = 3.5} x_j \le 200 $$
$$ \sum_{j \in J : l_j = 4.1} x_j \le 300 $$

### Обеспечение комплектности рам
Чтобы собрать $F$ рам, необходимо произвести как минимум $3F$ заготовок по 1.0 м и $2F$ заготовок по 1.5 м:
$$ \sum_{j \in J} a_j \cdot x_j \ge 3F $$
$$ \sum_{j \in J} b_j \cdot x_j \ge 2F $$

### Условия неотрицательности и целочисленности
$$ x_j \in \mathbb{Z}_+, \quad \forall j \in J $$
$$ F \in \mathbb{Z}_+ $$

In [4]:
from ortools.linear_solver import pywraplp


bar_stocks = {
    2.7: 100,
    3.5: 200,
    4.1: 300
}


req_1m = 3
req_1_5m = 2


patterns = []

for length, count in bar_stocks.items():
    max_1m = int(length // 1.0)
    max_1_5m = int(length // 1.5)
    
    for a in range(max_1m + 1):
        for b in range(max_1_5m + 1):
            if a * 1.0 + b * 1.5 <= length:

                if a > 0 or b > 0: 
                    patterns.append({
                        'length': length,
                        'a': a, 
                        'b': b  
                    })


solver = pywraplp.Solver.CreateSolver('SCIP')
if not solver:
    print("Ошибка")
    exit()


x = {}
for j in range(len(patterns)):
    x[j] = solver.IntVar(0, solver.infinity(), f'x_{j}')

F = solver.IntVar(0, solver.infinity(), 'Frames')


for length, stock in bar_stocks.items():
    solver.Add(
        sum(x[j] for j in range(len(patterns)) if patterns[j]['length'] == length) <= stock
    )


solver.Add(
    sum(patterns[j]['a'] * x[j] for j in range(len(patterns))) >= req_1m * F
)
solver.Add(
    sum(patterns[j]['b'] * x[j] for j in range(len(patterns))) >= req_1_5m * F
)

solver.Maximize(F)

status = solver.Solve()




if status == pywraplp.Solver.OPTIMAL:
    print("ОПТИМАЛЬНОЕ РЕШЕНИЕ НАЙДЕНО")
    print(f"Максимальное количество рам: {int(F.solution_value())}")
    
    total_1m = 0
    total_1_5m = 0
    
    print("\n--- ПЛАН РАСКРОЯ ---")
    for j in range(len(patterns)):
        val = x[j].solution_value()
        if val > 0:
            p = patterns[j]
            total_1m += p['a'] * val
            total_1_5m += p['b'] * val
            print(f"Брусок {p['length']} м -> {int(val)} шт. | "
                  f"Получаем: {p['a']} шт. по 1.0м и {p['b']} шт. по 1.5м")
            
    print("\n--- ИТОГО" \
    " ПРОИЗВЕДЕНО ЗАГОТОВОК ---")
    print(f"Заготовок 1.0 м: {int(total_1m)} (требуется {int(F.solution_value()) * req_1m})")
    print(f"Заготовок 1.5 м: {int(total_1_5m)} (требуется {int(F.solution_value()) * req_1_5m})")
    
    print("\n--- ОСТАТКИ БРУСКОВ НА СКЛАДЕ ---")
    for length, stock in bar_stocks.items():
        used = sum(x[j].solution_value() for j in range(len(patterns)) if patterns[j]['length'] == length)
        print(f"Бруски {length} м: использовано {int(used)} из {stock} (остаток {int(stock - used)})")
        
else:
    print("Решение не найдено.")

ОПТИМАЛЬНОЕ РЕШЕНИЕ НАЙДЕНО
Максимальное количество рам: 358

--- ПЛАН РАСКРОЯ ---
Брусок 2.7 м -> 100 шт. | Получаем: 1 шт. по 1.0м и 1 шт. по 1.5м
Брусок 3.5 м -> 200 шт. | Получаем: 2 шт. по 1.0м и 1 шт. по 1.5м
Брусок 4.1 м -> 208 шт. | Получаем: 1 шт. по 1.0м и 2 шт. по 1.5м
Брусок 4.1 м -> 92 шт. | Получаем: 4 шт. по 1.0м и 0 шт. по 1.5м

--- ИТОГО ПРОИЗВЕДЕНО ЗАГОТОВОК ---
Заготовок 1.0 м: 1076 (требуется 1074)
Заготовок 1.5 м: 716 (требуется 716)

--- ОСТАТКИ БРУСКОВ НА СКЛАДЕ ---
Бруски 2.7 м: использовано 100 из 100 (остаток 0)
Бруски 3.5 м: использовано 200 из 200 (остаток 0)
Бруски 4.1 м: использовано 300 из 300 (остаток 0)
